# Creating Training data for Semantic segmentation
Includes:  
- Tiling & Chipping  
- Batch & Pairing  
- Quality Check

In [1]:
# load libraries
from pathlib import Path
import geoai
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import geopandas as gpd
import rasterio
from rasterio.windows import Window

In [6]:
# raster and vector mask paths
ortho_path = "../data/raw/raster/vexcel/luzern-2025_epsg2056.tif"
mask_path = "../data/processed/combined_mask.tif"

In [7]:
# Verify raster alignment
with rasterio.open(ortho_path) as ortho:
    print("ORTHO")
    print(ortho.shape)
    print(ortho.crs)
    print(ortho.bounds)

with rasterio.open(mask_path) as mask:
    print("\nMASK")
    print(mask.shape)
    print(mask.crs)
    print(mask.bounds)

ORTHO
(13334, 13334)
EPSG:2056
BoundingBox(left=2666000.0, bottom=1209999.95, right=2667000.05, top=1211000.0)

MASK
(13334, 13334)
EPSG:2056
BoundingBox(left=2666000.0, bottom=1209999.95, right=2667000.05, top=1211000.0)


#### Subsetting area for smaller area
For the first run, i want to reduce the amount of data and therefore only look at a minimal amount of my original area of interest.

In [10]:
# Output files
ortho_subset = "../data/processed/ortho_subset_center.tif"
mask_subset = "../data/processed/mask_subset_center.tif"

crop_size = 1024

with rasterio.open(ortho_path) as src:

    # center vertically, left quarter horizontally
    col_center = src.width // 2
    row_center = src.height // 2

    window = Window(
        col_center - crop_size // 2,
        row_center - crop_size // 2,
        crop_size,
        crop_size
    )

    data = src.read(window=window)

    profile = src.profile.copy()
    profile.update(
        width=crop_size,
        height=crop_size,
        transform=rasterio.windows.transform(window, src.transform)
    )

    with rasterio.open(ortho_subset, "w", **profile) as dst:
        dst.write(data)

# Apply identical crop to mask
with rasterio.open(mask_path) as src:

    data = src.read(window=window)

    profile = src.profile.copy()
    profile.update(
        width=crop_size,
        height=crop_size,
        transform=rasterio.windows.transform(window, src.transform)
    )

    with rasterio.open(mask_subset, "w", **profile) as dst:
        dst.write(data)


#### Tiling and Chipping

In [ ]:
# Selecting tile size and stride
### starting with a smaller/moderarte overlap of 25% before scaling up
tile_size = (256, 256)  # Size of each tile (height, width)
stride = (192, 192)  # Stride for sliding window (height, width)

In [ ]:
# Generating tiles from the orthophoto and mask


#### Batch & Pairing

#### Quality & splitting